# NHLA 1-Side Classification

This notebook trains the single-face NHLA grading model. The workflow is intentionally:
1. prepare the extracted face CSV,
2. run Optuna tuning on a short validation pass,
3. train the final 1-side model using the best hyperparameters,
4. save metrics and checkpoints for comparison against the 2-side model.

In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_NAME = "26-8-18_75VAL"
# Validated faces: only boards whose grader validation grade equals the vision board grade,
# labelled with the per-face vision grade (the grader never validates individual faces).
USE_VALIDATED_FACES = True
CSV_NAME = "nhla_face_labels_validated.csv" if USE_VALIDATED_FACES else "nhla_face_labels.csv"
CSV_PATH = PROJECT_ROOT / "data" / DATASET_NAME / "nhla_dataset" / CSV_NAME
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = PROJECT_ROOT / "models" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"
OPTUNA_DIR = PROJECT_ROOT / "logs" / "optuna"
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics" / f"NHLA_Classification_1Side_{RUN_TIMESTAMP}"

# ------------------------------------------------------------------
# Configuration - change these knobs as needed
# ------------------------------------------------------------------
# Input geometry (keep identical to the 2-side notebook so both share one cache and the
# 1-side model_best.pt can initialise the 2-side backbone).
IMAGE_SCALE = 0.375
CANVAS = (512, 1888)
BATCH_SIZE = 16
NUM_WORKERS = 6
NUM_EPOCHS = 10           # resnet18: ~50 min/epoch on all 582k validated faces at this canvas
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.2
SEED = 42
CHECKPOINT_INTERVAL = 2

# Shared decode-once image cache (same cache as the 2-side notebook). None -> ~/.cache/nhla_images
IMAGE_CACHE_DIR = None

# Optuna tuning controls
OPTUNA_ENABLED = True
OPTUNA_TRIALS = 10
TUNE_EPOCHS = 4
TUNE_SUBSET_FRACTION = 0.1  # label-stratified fraction of boards per trial (~58k faces, ~5 min/epoch)
STUDY_NAME = f"nhla_1side_optuna_{RUN_TIMESTAMP}"
OPTUNA_DB_PATH = OPTUNA_DIR / "NHLA_Classification.db"
OPTUNA_SEARCH_SPACE = {
    "lr": {"low": 1e-5, "high": 1e-3, "log": True},
    "batch_size": [16, 32],
    "weight_decay": {"low": 1e-6, "high": 1e-2, "log": True},
    "encoder_name": ["resnet18", "resnet34"],  # resnet50 is ~2.3 h/epoch at this canvas
}

# Run naming
CHECKPOINT_NAME = f"NHLA_1Side_{RUN_TIMESTAMP}"
LOG_NAME = f"NHLA_1Side_{RUN_TIMESTAMP}"
METRICS_RUN_NAME = f"nhla_1side_{RUN_TIMESTAMP}"

# Model config
ENCODER_NAME = "resnet18"  # options: resnet18, resnet34, resnet50
PRETRAINED = True

print(f"CSV path: {CSV_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"TensorBoard dir: {TENSORBOARD_DIR}")
print(f"Metrics dir: {METRICS_DIR}")
print(f"Image scale: {IMAGE_SCALE}, canvas: {CANVAS}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL}")
print(f"Checkpoint name: {CHECKPOINT_NAME}")
print(f"Log name: {LOG_NAME}")
print(f"Metrics run name: {METRICS_RUN_NAME}")
print("Optuna search space:")
print(json.dumps(OPTUNA_SEARCH_SPACE, indent=2, default=str))

CSV path: /workspaces/Oak-Defect-Detection/data/26-8-18_75VAL/nhla_dataset/nhla_face_labels_validated.csv
Output dir: /workspaces/Oak-Defect-Detection/models/NHLA_Classification_1Side_20261009_165056
TensorBoard dir: /workspaces/Oak-Defect-Detection/logs/tensorboard/NHLA_Classification_1Side_20261009_165056
Metrics dir: /workspaces/Oak-Defect-Detection/data/metrics/NHLA_Classification_1Side_20261009_165056
Image scale: 0.375, canvas: (512, 1888)
Checkpoint interval: 2
Checkpoint name: NHLA_1Side_20261009_165056
Log name: NHLA_1Side_20261009_165056
Metrics run name: nhla_1side_20261009_165056
Optuna search space:
{
  "lr": {
    "low": 1e-05,
    "high": 0.001,
    "log": true
  },
  "batch_size": [
    16,
    32
  ],
  "weight_decay": {
    "low": 1e-06,
    "high": 0.01,
    "log": true
  },
  "encoder_name": [
    "resnet18",
    "resnet34"
  ]
}


In [2]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import notebooks.NHLA_Classification as nhla_module
import notebooks.NHLA_Classification_1Side as nhla_1side_module
importlib.reload(nhla_module)
importlib.reload(nhla_1side_module)
train_1side = nhla_1side_module.train_1side

result = train_1side(
    csv_path=str(CSV_PATH),
    output_dir=str(OUTPUT_DIR),
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_split=VAL_SPLIT,
    seed=SEED,
    optuna_trials=OPTUNA_TRIALS if OPTUNA_ENABLED else 0,
    tune_epochs=TUNE_EPOCHS,
    encoder_name=ENCODER_NAME,
    pretrained=PRETRAINED,
    search_space=OPTUNA_SEARCH_SPACE,
    study_name=STUDY_NAME,
    db_path=str(OPTUNA_DB_PATH),
    checkpoint_name=CHECKPOINT_NAME,
    log_name=LOG_NAME,
    metrics_run_name=METRICS_RUN_NAME,
    tensorboard_dir=str(TENSORBOARD_DIR),
    metrics_dir=str(METRICS_DIR),
    checkpoint_interval=CHECKPOINT_INTERVAL,
    num_workers=NUM_WORKERS,
    tune_subset_fraction=TUNE_SUBSET_FRACTION,
    image_cache_dir=IMAGE_CACHE_DIR,
    image_scale=IMAGE_SCALE,
    canvas=CANVAS,
)

print(json.dumps({
    "best_params": result["best_params"],
    "metrics": result["result"]["metrics"],
    "metrics_path": result["result"].get("metrics_path"),
}, indent=2))

2026-10-09 16:50:59.148106: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-09 16:50:59.164240: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-09 16:50:59.169476: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[I 2026-10-09 16:51:02,473] A new study created in RDB with name: nhla_1side_optuna_20261009_165056


Using device: cuda:0 (NVIDIA GeForce RTX 3090)


[I 2026-10-09 17:18:48,915] Trial 0 finished with value: 0.4370742385531752 and parameters: {'lr': 5.6115164153345e-05, 'batch_size': 16, 'weight_decay': 0.0002481040974867811, 'encoder_name': 'resnet18'}. Best is trial 0 with value: 0.4370742385531752.


Using device: cuda:0 (NVIDIA GeForce RTX 3090)


...

[I 2026-10-09 22:01:10,831] Trial 8 finished with value: 0.4304930927024218 and parameters: {'lr': 0.00012399967836846095, 'batch_size': 32, 'weight_decay': 0.001260466458564947, 'encoder_name': 'resnet18'}. Best is trial 3 with value: 0.4543163246122846.


Using device: cuda:0 (NVIDIA GeForce RTX 3090)


[I 2026-10-09 22:32:34,489] Trial 9 pruned. 


Optuna tuning complete for face mode. Best params: {'lr': 7.309539835912905e-05, 'batch_size': 32, 'weight_decay': 3.6138942712165278e-06, 'encoder_name': 'resnet34'}
Best 1-side Optuna params: {'lr': 7.309539835912905e-05, 'batch_size': 32, 'weight_decay': 3.6138942712165278e-06, 'encoder_name': 'resnet34'}
Using device: cuda:0 (NVIDIA GeForce RTX 3090)


{
  "best_params": {
    "lr": 7.309539835912905e-05,
    "batch_size": 32,
    "weight_decay": 3.6138942712165278e-06,
    "encoder_name": "resnet34"
  },
  "metrics": {
    "accuracy": 0.606786941580756,
    "balanced_accuracy": 0.5058383696833507,
    "macro_precision": 0.544770933257631,
    "macro_recall": 0.5058383696833507,
    "macro_f1": 0.5206085461695259,
    "weighted_precision": 0.6086493241394128,
    "weighted_recall": 0.606786941580756,
    "weighted_f1": 0.6049248773375996,
    "micro_f1": 0.606786941580756,
    "per_class_precision": [
      0.5030198210069731,
      0.7143693711885627,
      0.6563052445492045,
      0.43478260869565216,
      0.4207656612529002,
      0.5393828928524932
    ],
    "per_class_recall": [
      0.6129528652192822,
      0.6996202982241031,
      0.5782334999026543,
      0.3082614056720099,
      0.30392156862745096,
      0.5320405804546038
    ],
    "per_class_f1": [
      0.552571661213227,
      0.7069179122372312,
      0.6148007

## Notes

- Optuna is run before the final training step.
- The tuned parameters are saved to `best_params.json` in the output directory.
- Final training uses the best parameters rather than the initial defaults.